# 01 - Face Detection & Extraction

## Prerequisites
- Datasets must be available at configured paths (FF++, Celeb-DF, DFDC, HiDF)
- This is the **first notebook** in the pipeline — run it before any feature extraction

## Run Order
```
01_face_detection_extraction  (this notebook)
        |
        v
02, 03, 04  (run in parallel)
        |
        v
05_pipeline_tracking_and_manifest
```

## Outputs
- `preprocessed_faces_integrated/real|fake/...` — Cropped face images
- `integrated_nocuda.csv` — Combined dataset metadata

In [ ]:
!pip install --user timm imageio imageio-ffmpeg av

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import random
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import urllib.request

# Image processing
import cv2
from PIL import Image

# PyTorch (used for video frame sampling)
import torch

import site
import sys
sys.path.append(site.getusersitepackages())

# ============================================================================
# SET RANDOM SEED
# ============================================================================

def set_seed(seed=42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"{'='*70}")
print("ENVIRONMENT SETUP")
print(f"{'='*70}")
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"{'='*70}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Central configuration for the integrated training pipeline."""
    
    # ==========================================================================
    # DATA PERCENTAGE - Controls how much data to use from each dataset
    # ==========================================================================
    # Global default (used if per-dataset value not specified)
    # DATA_PERCENTAGE = 0.01  # 1% default
    
    # Per-dataset percentages (0.01 to 1.0)
    FFPP_PERCENTAGE = 0.01       # FaceForensics++ percentage
    CELEBDF_PERCENTAGE = 0.05  # Celeb-DF percentage
    DFDC_PERCENTAGE = 0.3      # DFDC percentage
    HIDF_PERCENTAGE = 0.1      # HiDF percentage
    
    # ==========================================================================
    # Dataset Paths (SWAN CERN Cloud)
    # ==========================================================================
    BASE_DIR = Path('.')  # Current directory (same as notebook)
    
    # Individual dataset paths (extra folder layer from zip extraction)
    FFPP_DIR = BASE_DIR / 'FF' / 'FaceForensics++'
    CELEBDF_DIR = BASE_DIR / 'CelebDF' / 'CelebDF'
    DFDC_DIR = BASE_DIR / 'DDFC' / 'deepfake-detection-challenge'
    HIDF_DIR = BASE_DIR / 'HiDF' / 'HiDF'
    
    # Output directories
    PREPROCESSED_DIR = BASE_DIR / 'preprocessed_faces_integrated'
    FEATURES_DIR = BASE_DIR / 'extracted_features_integrated'
    CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'
    RESULTS_DIR = BASE_DIR / 'results'
    INTEGRATED_METADATA_PATH = BASE_DIR / 'integrated_nocuda.csv'
    
    # Data settings
    FRAMES_PER_VIDEO = 20
    FACE_SIZE = 299
    TRAIN_RATIO = 0.8
    
    # Feature dimensions
    SPATIAL_DIM = 2048   # Xception output
    FREQ_DIM = 4         # FFT statistical features
    SEMANTIC_DIM = 768   # DINOv2 ViT-B/14 output
    
    # Training settings
    BATCH_SIZE = 128
    LEARNING_RATE = 1e-4
    EPOCHS = 20
    HIDDEN_DIM1 = 512
    HIDDEN_DIM2 = 256
    DROPOUT = 0.5
    FUSION_DIM = 512
    
    # Early stopping
    PATIENCE = 5
    MIN_DELTA = 0.001

config = Config()

# Create directory structure
def create_directories():
    """Create all necessary directories."""
    dirs = [
        config.PREPROCESSED_DIR / 'real',
        config.PREPROCESSED_DIR / 'fake',
        config.FEATURES_DIR / 'spatial',
        config.FEATURES_DIR / 'frequency',
        config.FEATURES_DIR / 'semantic',
        config.CHECKPOINTS_DIR,
        config.RESULTS_DIR
    ]
    for dir_path in dirs:
        dir_path.mkdir(parents=True, exist_ok=True)
    print("âœ“ Directory structure created")

create_directories()

# Display configuration
print(f"\n{'='*70}")
print("PROJECT CONFIGURATION")
print(f"{'='*70}")
print(f"Data Percentages:")
print(f"  - FaceForensics++: {config.FFPP_PERCENTAGE * 100:.1f}%")
print(f"  - Celeb-DF: {config.CELEBDF_PERCENTAGE * 100:.1f}%")
print(f"  - DFDC: {config.DFDC_PERCENTAGE * 100:.1f}%")
print(f"  - HiDF: {config.HIDF_PERCENTAGE * 100:.1f}%")
print(f"Base Directory: {config.BASE_DIR.absolute()}")
print(f"Frames per video: {config.FRAMES_PER_VIDEO}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Learning rate: {config.LEARNING_RATE}")
print(f"Epochs: {config.EPOCHS}")
print(f"Feature dimensions: Spatial={config.SPATIAL_DIM}, Freq={config.FREQ_DIM}, Semantic={config.SEMANTIC_DIM}")
print(f"{'='*70}")

## Section 2: Dataset Loaders

In [ ]:
# ============================================================================
# FACEFORENSICS++ DATASET LOADER
# ============================================================================

def load_ffpp_dataset(data_percentage=None):
    """
    Load FaceForensics++ dataset metadata.
    Uses CSV files from datasets/FaceForensics++/csv/
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset
    """
    # Use FFPP-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.FFPP_PERCENTAGE
    csv_dir = config.FFPP_DIR / 'csv'
    
    if not csv_dir.exists():
        print(f"âš  FaceForensics++ CSV directory not found: {csv_dir}")
        return pd.DataFrame()
    
    data = []
    
    # Load original (real) videos
    original_csv = csv_dir / 'original.csv'
    if original_csv.exists():
        df_orig = pd.read_csv(original_csv)
        # Sample the required percentage
        n_samples = max(1, int(len(df_orig) * data_percentage))
        df_orig_sampled = df_orig.sample(n=n_samples, random_state=42)
        
        for _, row in df_orig_sampled.iterrows():
            file_path = row['File Path']
            video_name = Path(file_path).stem
            full_path = config.FFPP_DIR / file_path
            if full_path.exists():
                data.append({
                    'video_id': f'FFPP_original_{video_name}',
                    'label': 'real',
                    'path': str(full_path),
                    'manipulation': 'Original',
                    'dataset': 'FaceForensics++'
                })
    
    # Load fake videos from different manipulation methods
    manipulation_csvs = ['Deepfakes.csv', 'Face2Face.csv', 'FaceSwap.csv', 
                         'NeuralTextures.csv', 'FaceShifter.csv']
    
    for csv_name in manipulation_csvs:
        csv_path = csv_dir / csv_name
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            manipulation = csv_name.replace('.csv', '')
            
            # Sample the required percentage
            n_samples = max(1, int(len(df) * data_percentage))
            df_sampled = df.sample(n=n_samples, random_state=42)
            
            for _, row in df_sampled.iterrows():
                file_path = row['File Path']
                video_name = Path(file_path).stem
                full_path = config.FFPP_DIR / file_path
                if full_path.exists():
                    data.append({
                        'video_id': f'FFPP_{manipulation}_{video_name}',
                        'label': 'fake',
                        'path': str(full_path),
                        'manipulation': manipulation,
                        'dataset': 'FaceForensics++'
                    })
    
    # Also check FF++_Metadata.csv for DeepFakeDetection videos
    metadata_csv = csv_dir / 'FF++_Metadata.csv'
    if metadata_csv.exists():
        df_meta = pd.read_csv(metadata_csv)
        # Filter for DeepFakeDetection
        df_dfd = df_meta[df_meta['File Path'].str.contains('DeepFakeDetection', na=False)]
        
        n_samples = max(1, int(len(df_dfd) * data_percentage))
        df_dfd_sampled = df_dfd.sample(n=min(n_samples, len(df_dfd)), random_state=42)
        
        for _, row in df_dfd_sampled.iterrows():
            file_path = row['File Path']
            video_name = Path(file_path).stem
            full_path = config.FFPP_DIR / file_path
            label = 'fake' if row['Label'] == 'FAKE' else 'real'
            if full_path.exists():
                data.append({
                    'video_id': f'FFPP_DeepFakeDetection_{video_name}',
                    'label': label,
                    'path': str(full_path),
                    'manipulation': 'DeepFakeDetection',
                    'dataset': 'FaceForensics++'
                })
    
    df_result = pd.DataFrame(data)
    print(f"âœ“ FaceForensics++: Loaded {len(df_result)} videos ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
    
    return df_result

In [ ]:
# ============================================================================
# CELEB-DF DATASET LOADER
# ============================================================================

def load_celebdf_dataset(data_percentage=None):
    """
    Load Celeb-DF dataset metadata.
    Uses List_of_testing_videos.txt for ground truth labels.
    
    Format: <label> <path>
    - Label 1 = Real
    - Label 0 = Fake (Celeb-synthesis)
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset
    """
    # Use Celeb-DF-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.CELEBDF_PERCENTAGE
    list_file = config.CELEBDF_DIR / 'List_of_testing_videos.txt'
    
    if not list_file.exists():
        print(f"âš  Celeb-DF list file not found: {list_file}")
        return pd.DataFrame()
    
    data = []
    
    with open(list_file, 'r') as f:
        lines = f.readlines()
    
    # Parse all entries
    all_entries = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) >= 2:
            file_label = int(parts[0])
            file_path = parts[1]
            all_entries.append((file_label, file_path))
    
    # Sample the required percentage
    n_samples = max(1, int(len(all_entries) * data_percentage))
    random.seed(42)
    sampled_entries = random.sample(all_entries, min(n_samples, len(all_entries)))
    
    for file_label, file_path in sampled_entries:
        video_name = Path(file_path).stem
        folder_name = Path(file_path).parent.name
        full_path = config.CELEBDF_DIR / file_path
        
        # Label 1 = Real, Label 0 = Fake
        label = 'real' if file_label == 1 else 'fake'
        
        # Determine manipulation type based on folder
        if 'synthesis' in folder_name.lower():
            manipulation = 'Celeb-synthesis'
        elif 'real' in folder_name.lower():
            manipulation = 'Original'
        else:
            manipulation = folder_name
        
        if full_path.exists():
            data.append({
                'video_id': f'CelebDF_{folder_name}_{video_name}',
                'label': label,
                'path': str(full_path),
                'manipulation': manipulation,
                'dataset': 'Celeb-DF'
            })
    
    df_result = pd.DataFrame(data)
    print(f"âœ“ Celeb-DF: Loaded {len(df_result)} videos ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
    
    return df_result

In [ ]:
# ============================================================================
# DFDC (DEEPFAKE DETECTION CHALLENGE) DATASET LOADER
# ============================================================================

def load_dfdc_dataset(data_percentage=None):
    """
    Load DFDC dataset metadata.
    Uses metadata.json from train_sample_videos directory.
    
    Format: {"filename.mp4": {"label": "REAL"/"FAKE", "split": "train", "original": ...}}
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset
    """
    # Use DFDC-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.DFDC_PERCENTAGE
    videos_dir = config.DFDC_DIR / 'train_sample_videos'
    metadata_file = videos_dir / 'metadata.json'
    
    if not metadata_file.exists():
        print(f"âš  DFDC metadata file not found: {metadata_file}")
        return pd.DataFrame()
    
    data = []
    
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    # Get all entries
    all_entries = list(metadata.items())
    
    # Sample the required percentage
    n_samples = max(1, int(len(all_entries) * data_percentage))
    random.seed(42)
    sampled_entries = random.sample(all_entries, min(n_samples, len(all_entries)))
    
    for filename, info in sampled_entries:
        video_path = videos_dir / filename
        video_name = Path(filename).stem
        label = 'real' if info['label'] == 'REAL' else 'fake'
        
        # Determine manipulation type
        if label == 'fake':
            manipulation = 'DFDC_DeepFake'
        else:
            manipulation = 'Original'
        
        if video_path.exists():
            data.append({
                'video_id': f'DFDC_{video_name}',
                'label': label,
                'path': str(video_path),
                'manipulation': manipulation,
                'dataset': 'DFDC'
            })
    
    df_result = pd.DataFrame(data)
    print(f"âœ“ DFDC: Loaded {len(df_result)} videos ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
    
    return df_result

In [ ]:
# ============================================================================
# HiDF DATASET LOADER
# ============================================================================

def load_hidf_dataset(data_percentage=None):
    """
    Load HiDF dataset metadata using FOLDER-BASED LABELING.
    
    Labeling based on folder names:
    - Files in Fake-vid, Fake-img folders â†’ label = "fake"
    - Files in Real-vid, Real-img folders â†’ label = "real"
    
    This approach directly scans the folders instead of relying on metadata.csv,
    which ensures accurate labeling based on actual file locations.
    
    Directories:
    - Real-vid: Real videos (.mp4)
    - Real-img: Real images (.jpg, .png) - may not exist yet
    - Fake-vid: Fake videos (.mp4)
    - Fake-img: Fake images (.jpg, .png)
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset, media_type
    """
    # Use HiDF-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.HIDF_PERCENTAGE
    
    data = []
    
    # Define folder configurations: (folder_name, label, media_type, manipulation)
    folder_configs = [
        ('Fake-vid', 'fake', 'vid', 'HiDF_FaceSwap'),
        ('Fake-img', 'fake', 'img', 'HiDF_FaceSwap'),
        ('Real-vid', 'real', 'vid', 'Original'),
        ('Real-img', 'real', 'img', 'Original'),
    ]
    
    for folder_name, label, media_type, manipulation in folder_configs:
        folder_path = config.HIDF_DIR / folder_name
        
        if not folder_path.exists():
            print(f"  âš  HiDF folder not found: {folder_name}")
            continue
        
        # Define file extensions based on media type
        if media_type == 'vid':
            extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv']
        else:
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
        
        # Collect all files from this folder
        all_files = []
        for ext in extensions:
            all_files.extend(list(folder_path.glob(ext)))
        
        if not all_files:
            print(f"  âš  No files found in {folder_name}")
            continue
        
        # Sample the required percentage
        n_samples = max(1, int(len(all_files) * data_percentage))
        random.seed(42)
        sampled_files = random.sample(all_files, min(n_samples, len(all_files)))
        
        # Add sampled files to data
        for file_path in sampled_files:
            file_name = file_path.stem.strip()
            data.append({
                'video_id': f'HiDF_{label}_{folder_name}_{file_name}',
                'label': label,
                'path': str(file_path),
                'manipulation': manipulation,
                'dataset': 'HiDF',
                'media_type': media_type
            })
    
    df_result = pd.DataFrame(data)
    print(f"âœ“ HiDF: Loaded {len(df_result)} items ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
        if 'media_type' in df_result.columns:
            vid_count = len(df_result[df_result['media_type'] == 'vid'])
            img_count = len(df_result[df_result['media_type'] == 'img'])
            print(f"  Videos: {vid_count}, Images: {img_count}")
    
    return df_result

In [ ]:
# ============================================================================
# LOAD AND COMBINE ALL DATASETS
# ============================================================================

def load_all_datasets():
    """
    Load and combine all four datasets.
    Each dataset uses its own percentage from Config.
    
    Returns:
        DataFrame with combined metadata from all datasets
    """
    print(f"\n{'='*70}")
    print("LOADING DATASETS (per-dataset percentages)")
    print(f"{'='*70}")
    
    # Load each dataset (each uses its own config percentage)
    df_ffpp = load_ffpp_dataset()
    df_celebdf = load_celebdf_dataset()
    df_dfdc = load_dfdc_dataset()
    df_hidf = load_hidf_dataset()
    
    # Combine all datasets
    all_dfs = [df for df in [df_ffpp, df_celebdf, df_dfdc, df_hidf] if len(df) > 0]
    
    if not all_dfs:
        print("âš  No datasets loaded!")
        return pd.DataFrame()
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    # Remove duplicates based on path
    combined_df = combined_df.drop_duplicates(subset=['path'])
    
    print(f"\n{'='*70}")
    print("COMBINED DATASET STATISTICS")
    print(f"{'='*70}")
    print(f"Total samples: {len(combined_df)}")
    print(f"Real: {len(combined_df[combined_df['label'] == 'real'])}, Fake: {len(combined_df[combined_df['label'] == 'fake'])}")
    print(f"\nSamples per dataset:")
    print(combined_df['dataset'].value_counts())
    print(f"\nManipulation types:")
    print(combined_df['manipulation'].value_counts())
    
    # Save combined metadata
    combined_df.to_csv(config.INTEGRATED_METADATA_PATH, index=False)
    print(f"\nâœ“ Metadata saved to: {config.INTEGRATED_METADATA_PATH}")
    
    return combined_df

# Load all datasets
integrated_metadata = load_all_datasets()

## Section 3: Face Detection & Extraction

In [ ]:
# ============================================================================
# FACE EXTRACTION (skips already-processed items automatically)
# ============================================================================

# Download OpenCV DNN face detector model files
FACE_PROTO = "deploy.prototxt"
FACE_MODEL = "res10_300x300_ssd_iter_140000_fp16.caffemodel"
PROTO_URL = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
MODEL_URL = "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20180205_fp16/res10_300x300_ssd_iter_140000_fp16.caffemodel"

if not os.path.exists(FACE_PROTO):
    print("Downloading face detector prototxt...")
    urllib.request.urlretrieve(PROTO_URL, FACE_PROTO)
if not os.path.exists(FACE_MODEL):
    print("Downloading face detector model...")
    urllib.request.urlretrieve(MODEL_URL, FACE_MODEL)

face_net = cv2.dnn.readNetFromCaffe(FACE_PROTO, FACE_MODEL)
print("âœ“ OpenCV DNN face detector loaded")


# ============================================================================
# ROBUST VIDEO READING WITH FALLBACK BACKENDS
# cv2.VideoCapture often fails on SWAN CERN due to missing FFmpeg codecs.
# We try: cv2 â†’ imageio â†’ av (PyAV) in that order.
# ============================================================================

def _read_frames_cv2(file_path, num_frames):
    """Try reading video frames with OpenCV."""
    cap = cv2.VideoCapture(str(file_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None
    frame_indices = torch.linspace(0, total_frames - 1, num_frames).long()
    frames = []
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx.item())
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames if frames else None


def _read_frames_imageio(file_path, num_frames):
    """Try reading video frames with imageio + imageio-ffmpeg."""
    try:
        import imageio.v3 as iio
        frames_all = iio.imread(str(file_path), plugin="pyav")
        total = len(frames_all)
        if total == 0:
            return None
        indices = torch.linspace(0, total - 1, min(num_frames, total)).long().tolist()
        return [frames_all[i] for i in indices]
    except Exception:
        pass
    try:
        import imageio
        reader = imageio.get_reader(str(file_path), format='ffmpeg')
        total = reader.count_frames()
        if total == 0:
            reader.close()
            return None
        indices = set(torch.linspace(0, total - 1, min(num_frames, total)).long().tolist())
        frames = []
        for i, frame in enumerate(reader):
            if i in indices:
                frames.append(frame)  # Already RGB
            if len(frames) >= num_frames:
                break
        reader.close()
        return frames if frames else None
    except Exception:
        return None


def _read_frames_av(file_path, num_frames):
    """Try reading video frames with PyAV."""
    try:
        import av
        container = av.open(str(file_path))
        stream = container.streams.video[0]
        total = stream.frames
        if total == 0:
            # Try counting manually
            total = sum(1 for _ in container.decode(video=0))
            container.close()
            container = av.open(str(file_path))
            if total == 0:
                container.close()
                return None
        indices = set(torch.linspace(0, max(total - 1, 0), min(num_frames, max(total, 1))).long().tolist())
        frames = []
        for i, frame in enumerate(container.decode(video=0)):
            if i in indices:
                frames.append(frame.to_ndarray(format='rgb24'))
            if len(frames) >= num_frames:
                break
        container.close()
        return frames if frames else None
    except Exception:
        return None


# Detect which video backend works (tested once at startup)
_VIDEO_BACKEND = None

def _detect_video_backend(test_path=None):
    """Test which video reading backend works on this system."""
    global _VIDEO_BACKEND
    if _VIDEO_BACKEND is not None:
        return _VIDEO_BACKEND

    # If no test video provided, just check module availability
    backends = []
    
    # Check cv2 video support
    try:
        cap = cv2.VideoCapture()
        backends.append('cv2')
    except Exception:
        pass
    
    try:
        import imageio
        backends.append('imageio')
    except ImportError:
        pass
    
    try:
        import av
        backends.append('av')
    except ImportError:
        pass
    
    print(f"  Available video backends: {backends}")
    
    if test_path:
        # Actually test with a real video
        for backend_name, read_fn in [('cv2', _read_frames_cv2), ('imageio', _read_frames_imageio), ('av', _read_frames_av)]:
            if backend_name not in backends:
                continue
            result = read_fn(test_path, 2)
            if result and len(result) > 0:
                print(f"  âœ“ Working video backend: {backend_name}")
                _VIDEO_BACKEND = backend_name
                return _VIDEO_BACKEND
            else:
                print(f"  âœ— {backend_name} failed to read test video")
        
        print("  âš  No video backend could read the test video!")
        _VIDEO_BACKEND = 'none'
    else:
        _VIDEO_BACKEND = backends[0] if backends else 'none'
    
    return _VIDEO_BACKEND


def read_video_frames(file_path, num_frames=20):
    """
    Read frames from a video file using the best available backend.
    Falls back through cv2 â†’ imageio â†’ av until one works.
    
    Returns:
        List of RGB numpy arrays, or empty list if all backends fail.
    """
    for read_fn in [_read_frames_cv2, _read_frames_imageio, _read_frames_av]:
        result = read_fn(file_path, num_frames)
        if result and len(result) > 0:
            return result
    return []


def detect_and_crop_face(image_rgb, image_size=299, confidence_threshold=0.5):
    """Detect and crop the largest face using OpenCV DNN. Returns PIL Image or None."""
    h, w = image_rgb.shape[:2]
    blob = cv2.dnn.blobFromImage(image_rgb, 1.0, (300, 300), (104.0, 177.0, 123.0))
    face_net.setInput(blob)
    detections = face_net.forward()

    best_face = None
    best_conf = confidence_threshold

    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > best_conf:
            x1 = max(0, int(detections[0, 0, i, 3] * w))
            y1 = max(0, int(detections[0, 0, i, 4] * h))
            x2 = min(w, int(detections[0, 0, i, 5] * w))
            y2 = min(h, int(detections[0, 0, i, 6] * h))
            # Add margin
            margin = int(0.15 * max(x2 - x1, y2 - y1))
            x1 = max(0, x1 - margin)
            y1 = max(0, y1 - margin)
            x2 = min(w, x2 + margin)
            y2 = min(h, y2 + margin)
            best_face = (x1, y1, x2, y2)
            best_conf = conf

    if best_face is None:
        return None

    x1, y1, x2, y2 = best_face
    face_crop = image_rgb[y1:y2, x1:x2]
    face_pil = Image.fromarray(face_crop).resize((image_size, image_size), Image.LANCZOS)
    return face_pil


def extract_faces_integrated(metadata_df, output_dir, frames_per_video=20, image_size=299):
    """
    Extract faces from videos/images using OpenCV DNN face detector.
    Handles both video and image inputs.
    Uses fallback video backends (cv2 â†’ imageio â†’ av) for SWAN CERN compatibility.
    AUTOMATICALLY SKIPS already-processed items based on output directory existence.

    Args:
        metadata_df: DataFrame with video/image metadata
        output_dir: Directory to save extracted faces
        frames_per_video: Number of frames to sample from videos
        image_size: Output face image size
    """
    print(f"\n{'='*70}")
    print("FACE EXTRACTION WITH OPENCV DNN")
    print(f"{'='*70}")

    # ---- Diagnostic: find a test video and check which backend works ----
    test_video = None
    for _, row in metadata_df.iterrows():
        ext = Path(row['path']).suffix.lower()
        if ext in ['.mp4', '.avi', '.mov', '.mkv']:
            if Path(row['path']).exists():
                test_video = row['path']
                break

    if test_video:
        print(f"\n  Testing video reading with: {Path(test_video).name}")
        backend = _detect_video_backend(test_video)
        if backend == 'none':
            print("  âš  WARNING: No video backend works! Video files will be skipped.")
            print("  Try: pip install imageio imageio-ffmpeg av")
    else:
        print("  No video files found in metadata, skipping video backend test.")
        _detect_video_backend()

    processed = 0
    skipped = 0
    errors = 0
    video_errors = 0

    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Extracting faces"):
        file_path = row['path']
        video_id = row['video_id']
        label = row['label']
        media_type = row.get('media_type', 'vid')  # Default to video

        # Determine media type from file extension if not specified
        ext = Path(file_path).suffix.lower()
        if ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            media_type = 'img'
        elif ext in ['.mp4', '.avi', '.mov', '.mkv']:
            media_type = 'vid'

        # Sanitize video_id: strip whitespace to avoid path errors
        video_id = str(video_id).strip()
        video_output_dir = output_dir / label / video_id

        # Skip if already processed (key logic for re-runs)
        if video_output_dir.exists():
            existing_files = list(video_output_dir.glob('*.png'))
            min_expected = 1 if media_type == 'img' else max(1, frames_per_video // 4)
            if len(existing_files) >= min_expected:
                skipped += 1
                continue

        video_output_dir.mkdir(parents=True, exist_ok=True)

        try:
            if media_type == 'img':
                # Handle image
                img = cv2.imread(file_path)
                if img is None:
                    errors += 1
                    continue

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                face_pil = detect_and_crop_face(img_rgb, image_size)

                if face_pil is not None:
                    save_path = video_output_dir / 'frame_0.png'
                    face_pil.save(str(save_path))
                    processed += 1
                else:
                    errors += 1

            else:
                # Handle video using robust multi-backend reader
                frames_rgb = read_video_frames(file_path, frames_per_video)

                if not frames_rgb:
                    video_errors += 1
                    errors += 1
                    if video_errors <= 3:
                        print(f"\n  âš  Video read failed: {Path(file_path).name}")
                    elif video_errors == 4:
                        print(f"\n  âš  Suppressing further video error messages...")
                    continue

                saved_count = 0
                for frame_rgb in frames_rgb:
                    face_pil = detect_and_crop_face(frame_rgb, image_size)

                    if face_pil is not None:
                        save_path = video_output_dir / f'frame_{saved_count}.png'
                        face_pil.save(str(save_path))
                        saved_count += 1

                    if saved_count >= frames_per_video:
                        break

                if saved_count > 0:
                    processed += 1
                else:
                    errors += 1

        except Exception as e:
            errors += 1

    print(f"\n{'='*70}")
    print(f"Face extraction complete!")
    print(f"Processed: {processed}, Skipped (already done): {skipped}, Errors: {errors}")
    if video_errors > 0:
        print(f"  âš  Video read failures: {video_errors} (try: pip install imageio imageio-ffmpeg av)")
    print(f"{'='*70}")

# Run face extraction
extract_faces_integrated(
    integrated_metadata, 
    config.PREPROCESSED_DIR,
    config.FRAMES_PER_VIDEO, 
    config.FACE_SIZE)

## Validation

In [ ]:
# ============================================================================
# QUICK VALIDATION
# ============================================================================

print(f"\n{'='*70}")
print("VALIDATION - Face Extraction Results")
print(f"{'='*70}")

total_items = 0
total_faces = 0
for label in ["real", "fake"]:
    label_dir = config.PREPROCESSED_DIR / label
    if not label_dir.exists():
        print(f"  {label}: directory not found!")
        continue
    items = [d for d in label_dir.iterdir() if d.is_dir()]
    faces = sum(sum(1 for f in os.listdir(d) if f.endswith(".png")) for d in items)
    print(f"  {label}: {len(items)} items, {faces} face images")
    total_items += len(items)
    total_faces += faces

print(f"  TOTAL: {total_items} items, {total_faces} face images")

# Check metadata CSV
meta_path = config.INTEGRATED_METADATA_PATH
if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n  integrated_nocuda.csv: {len(df)} rows")
else:
    print(f"\n  WARNING: {meta_path} not found!")

if total_faces > 0:
    print(f"\nFace extraction successful!")
    print("Next step: Run 02, 03, 04 notebooks in parallel.")
else:
    print(f"\nNo faces extracted. Check dataset paths and logs above.")